# NFDI4Objects KG — Ogham Sites by County (local variant)

## About this notebook

This notebook introduces the
[NFDI4Objects Knowledge Graph (N4O KG)](https://graph.nfdi4objects.net/)
— a research-oriented triple store that integrates data about
material cultural heritage from many partner repositories across
Germany. We query it with SPARQL and count how many
[Ogham stones](https://en.wikipedia.org/wiki/Ogham_inscription) are
known from each Irish county, then visualise the distribution as a
pie chart and scatter plot.

It is part of an Open Educational Resource (OER) series on knowledge
graphs and linked open data. It is designed to stand on its own:
you do not need to have read any other notebook in the series to
follow along. A browser-executable variant of this notebook is
available as `n4okg-ogham-sites-county-live.qmd` (same content,
runs in the browser via Pyodide without any local Python setup).

### Why this dataset?

The Ogham corpus is one of several datasets hosted as a *named
collection* inside the N4O KG (here:
[`collection/9`](https://graph.nfdi4objects.net/collection/9)). It
is useful for teaching because:

- **The ontology is compact and domain-specific**
  ([ontology.ogham.link](http://ontology.ogham.link/)), so queries
  read almost like plain English once you know the three or four
  relevant classes (`OghamSite`, `OghamStone_CIIC`, `County`).
- **It exposes aggregation patterns well**: counting stones per
  county requires a `GROUP BY` with a `COUNT(DISTINCT …)` — the
  most common workhorse pattern in SPARQL analytics.
- **It is small** (around two dozen counties, a few hundred
  stones), so the whole dataset fits in memory easily.

### Data-context notes

N4O KG data is contributed by different research projects under
shared infrastructure; every project brings its own ontology. The
Ogham ontology (`oghamonto:`) is separate from the NFDI4Objects
core vocabulary (`n4o:`). This is normal for domain-rich knowledge
graphs and one of the reasons SPARQL is expressive: you can mix
vocabularies freely in a single query.

### Requirements

```bash
pip install SPARQLWrapper pandas matplotlib
```

### Tooling notes

- **`SPARQLWrapper`** is the de-facto standard Python wrapper for
  SPARQL endpoints outside the browser; it handles the HTTP
  negotiation and result parsing for us.
- **`pandas`** holds the flattened results; once the data is in a
  DataFrame, standard data-science tooling applies.
- **`matplotlib`** for static visualisations. For maps of the same
  dataset, see the companion notebook `n4okg-ogham-sites-map.ipynb`.

## Step 1 — Defining the SPARQL query

The query asks for every instance of `oghamonto:OghamSite`, looks
up the `County` it lies in (via `oghamonto:within`), and counts how
many catalogued Ogham stones (`oghamonto:OghamStone_CIIC`) have been
disclosed at that site. We group by county.

Two observations on query design that generalise beyond this
dataset:

- **`COUNT(DISTINCT ?stone)`** — not just `COUNT(?stone)`. A stone
  could in principle be linked through more than one path;
  `DISTINCT` makes sure we count each stone once.
- **Full predicate URIs for `rdf:type`**: we spell
  `<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>` rather than
  using SPARQL's `a` shorthand here. Both work; the full URI is
  sometimes preferred in educational contexts because it exposes
  the underlying mechanics.

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 120

SPARQL_ENDPOINT = "https://graph.nfdi4objects.net/api/sparql"

SPARQL_QUERY = """
PREFIX oghamonto: <http://ontology.ogham.link/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?county (count(distinct ?stone) as ?count) WHERE {
  ?item <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://ontology.ogham.link/OghamSite> .
  ?item oghamonto:within ?c .
  ?c a oghamonto:County .
  ?c rdfs:label ?county .
  ?stone oghamonto:disclosedAt ?item .
  ?stone a oghamonto:OghamStone_CIIC .
} GROUP BY ?county ORDER BY DESC(?count)
"""

def fetch_sparql(query, endpoint=SPARQL_ENDPOINT):
    sparql = SPARQLWrapper(endpoint)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    return sparql.queryAndConvert()["results"]["bindings"]

print("✓ Helpers defined. Run the next cell to load the data.")

## Step 2 — Loading the data

SPARQL results come back as JSON in a format called *bindings*: a list
of dictionaries, one per solution, where each key maps to a
`{"type": ..., "value": ...}` object. We flatten these into plain
records and build a DataFrame.

In [ ]:
results = fetch_sparql(SPARQL_QUERY)

rows = []
for r in results:
    rows.append({
        "county": r["county"]["value"],
        "count":  int(r["count"]["value"]),
    })

df = pd.DataFrame(rows)
assert not df.empty, "⚠ Query returned zero rows — check endpoint and query."
print(f"✓ {len(df)} counties loaded (total stones: {df['count'].sum()})")
df

## Step 3 — Visualising the distribution

### Pie chart with an 'Other' bucket

A pie chart is quick to read for a dozen or so categories but breaks
down when many slices are tiny. We group every county that contributes
less than 3 % of the total into a combined 'Other' wedge, and label
each slice with both its percentage and the raw stone count.

In [ ]:
assert 'df' in dir() and not df.empty, "⚠ Please run the data-loading cell first."

total_count = df["count"].sum()
df_pct = df.assign(percentage=df["count"] / total_count * 100)

major = df_pct[df_pct["percentage"] >= 3].copy()
minor = df_pct[df_pct["percentage"] < 3]

if not minor.empty:
    other_row = pd.DataFrame([{
        "county":     "Other",
        "count":      minor["count"].sum(),
        "percentage": minor["percentage"].sum(),
    }])
    major = pd.concat([major, other_row], ignore_index=True)

major = major.sort_values("count", ascending=False)

def make_autopct(values):
    def _autopct(pct):
        count = int(round(pct * sum(values) / 100.0))
        return f"{pct:.1f}% ({count:d})"
    return _autopct

fig, ax = plt.subplots(figsize=(9, 9))
major.set_index("county")["count"].plot(
    kind="pie",
    ax=ax,
    autopct=make_autopct(major["count"]),
    startangle=90,
    colors=plt.cm.Paired.colors,
)
ax.set_title("Distribution of Ogham sites by county (<3 % grouped as 'Other')",
             fontsize=13)
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### Scatter plot

A scatter plot per county makes the long tail of low-count counties
easier to see than the pie chart does. We show the raw count with
the county label annotated.

In [ ]:
assert 'df' in dir() and not df.empty, "⚠ Please run the data-loading cell first."

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(df))
ax.scatter(x, df["count"], color="steelblue", s=60)
ax.set_title("Stone count per county")
ax.set_xlabel("County")
ax.set_ylabel("Stone count")
ax.set_xticks(x)
ax.set_xticklabels([f"{c} [{n}]" for c, n in zip(df["county"], df["count"])],
                   rotation=45, ha="right")
ax.grid(color="lightgrey", linestyle="--", linewidth=0.5)
plt.tight_layout()
plt.show()

## Step 4 — Exploring the data

The cells below are a free playground — try answering questions of
your own by modifying the DataFrame or the SPARQL query.

In [ ]:
assert 'df' in dir(), "⚠ Please run the data-loading cell first."

# Example: the five counties with the most Ogham stones.
print("Top-five counties by stone count:")
df.head(5)

In [ ]:
assert 'df' in dir(), "⚠ Please run the data-loading cell first."

# Example: how many counties account for 80 % of all stones?
cumulative = df["count"].cumsum() / df["count"].sum()
n80 = int((cumulative < 0.80).sum()) + 1
print(f"{n80} counties out of {len(df)} account for 80 % of all Ogham stones.")

---

*This notebook is part of the Open Educational Resources of
[NFDI4Objects](https://www.nfdi4objects.net/).*